# TabICLv2 (Inria) Serving endpoints

Demonstrates how to provision and use a GPU serving endpoint with **TabICLv2** for real-time inference. 

This allows for data analyst and data scientist to have a universal classifier/regressor/forecaster available without having to manage GPU infrastructure. The downside is the [limitation on the serving endpoints](https://docs.databricks.com/aws/en/machine-learning/model-serving/model-serving-limits), specially on payload MB limits. 

**Compute:** Serverless CPU environment v5. The serving endpoint GPU is used for inference. 

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.

In [0]:
%pip install tabicl==2.1.1 --quiet

In [0]:
dbutils.library.restartPython()

## Configuration

Catalog/schema must match the shared data-prep notebook. `common/` is added to the
path so we can import the shared evaluation helpers.


In [0]:
import os, sys

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation, imported from common/config.py)
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# MLflow experiment (shared naming convention across vendors)
current_user = spark.sql("SELECT current_user()").collect()[0][0]
# Serving endpoints mlflow is to register the unverisal TabICL models
MLFLOW_EXPERIMENT_NAME_SERVING_ENDPOINTS = f"/Users/{current_user}/tabular-fm-databricks-serving-endpoints"
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"common/ path:   {COMMON_PATH}  (exists={os.path.isdir(COMMON_PATH)})")

## Import Libraries

In [0]:
import json
import time

import numpy as np
import pandas as pd
import mlflow
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec, DataType, ParamSchema, ParamSpec

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

import torch
from tabicl import TabICLClassifier

from evaluation import (
    split_xy, classification_metrics, train_baselines_classification,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME_SERVING_ENDPOINTS)



## Create universal serving endpoint for classification

The created serving endpoint holds a TabICL model without context data, which allows users to pass both training (i.e. context) samples and test samples to infer. This is useful to quickly obtain a prediction based on any dataset.

### Create Pyfunc MLflow model

In [0]:
class TabICLClassifierUniversal(mlflow.pyfunc.PythonModel):
    """
    Universal PyFunc wrapper for TabICL that accepts any classification dataset at inference time.
    Rows marked "train" via the `__role__` column serve as in-context examples; rows marked "test" are classified.
    Returns predicted labels and per-class probabilities (JSON-encoded) with a fixed output schema regardless of class count.

    Same input conventions as TabICLClassifierModel:
        - Column `__role__`: "train" or "test"
        - Column `__target__`: label for train rows
        - All other columns: numeric features

    Output schema is FIXED for any number of classes:
        - prediction: str (predicted class label)
        - probabilities: str (JSON object mapping class_label → probability)
    """

    def load_context(self, context):
        import torch
        from tabicl import TabICLClassifier

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clf = TabICLClassifier(device=device)

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        params = params or {}
        role_col = params.get("role_column", "__role__")
        target_col = params.get("target_column", "__target__")

        train_mask = model_input[role_col] == "train"
        test_mask = model_input[role_col] == "test"

        feature_cols = [c for c in model_input.columns if c not in (role_col, target_col)]

        X_train = model_input.loc[train_mask, feature_cols].values.astype(np.float32)
        y_train = model_input.loc[train_mask, target_col].values
        X_test = model_input.loc[test_mask, feature_cols].values.astype(np.float32)

        self.clf.fit(X_train, y_train)
        predictions = self.clf.predict(X_test)
        probabilities = self.clf.predict_proba(X_test)

        # Get class labels in the same order as predict_proba columns
        classes = self.clf.classes_

        # Encode probabilities as JSON: {"class_0": 0.85, "class_1": 0.15, ...}
        prob_dicts = [
            json.dumps({str(cls): round(float(p), 6) for cls, p in zip(classes, row)})
            for row in probabilities
        ]

        return pd.DataFrame({
            "prediction": predictions,
            "probabilities": prob_dicts,  # JSON string — works for 2, 5, or 100 classes
        })


### Register model into UC

In [0]:
# --- Register universal classifier with a FIXED output schema (works for any number of classes) ---
registered_universal_name = f"{CATALOG}.{SCHEMA}.tabicl_classifier_universal"

output_schema = Schema([
    ColSpec(DataType.string, "prediction"),
    ColSpec(DataType.string, "probabilities"),  # JSON object: class → prob
])
param_schema = ParamSchema([
    ParamSpec("role_column", DataType.string, "__role__"),
    ParamSpec("target_column", DataType.string, "__target__"),
])
signature = ModelSignature(inputs=None, outputs=output_schema, params=param_schema)

# Multiclass input example (iris 3-class)
input_example = pd.DataFrame({
    "f0": [5.1, 4.9, 7.0, 6.3],
    "f1": [3.5, 3.0, 3.2, 3.3],
    "f2": [1.4, 1.4, 4.7, 6.0],
    "f3": [0.2, 0.2, 1.4, 2.5],
    "__role__": ["train", "train", "train", "test"],
    "__target__": ["setosa", "setosa", "versicolor", ""],
})

with mlflow.start_run(run_name="tabicl_classifier_universal") as run:
    model_info = mlflow.pyfunc.log_model(
        name="tabicl_classifier_universal",
        python_model=TabICLClassifierUniversal(),
        signature=signature,
        input_example=input_example,
        pip_requirements=["tabicl", "pandas", "numpy"],
        registered_model_name=registered_universal_name,
    )
    print(f"Model URI: {model_info.model_uri}")
    print(f"Registered as: {registered_universal_name}")
    print(f"\nOutput schema is class-count agnostic:")
    print(f"  prediction: string (class label)")
    print(f"  probabilities: string (JSON dict of class → prob)")
    print(f"\nConsumer parses probabilities with: json.loads(row['probabilities'])")

### Create GPU serving endpoint

In [0]:
w = WorkspaceClient()
client = mlflow.MlflowClient()

In [0]:
serving_endpoint_universal = "tabicl_classifier_universal"

versions = client.search_model_versions(f"name='{registered_universal_name}'")
latest_version = str(max(int(v.version) for v in versions))
print(f"Deploying model version {latest_version} of '{registered_universal_name}'")

endpoint = w.serving_endpoints.create(
    name=serving_endpoint_universal,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=registered_universal_name,
                entity_version=latest_version,
                workload_size="Small",
                workload_type=ServingModelWorkloadType.GPU_SMALL,
                scale_to_zero_enabled=True,
            )
        ]
    ),
)
print(f"Endpoint '{serving_endpoint_universal}' creation triggered.")

In [0]:
# Poll until the serving endpoint is ready
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate

print(f"Waiting for endpoint '{serving_endpoint_universal}' to be ready...")
while True:
    endpoint_state = w.serving_endpoints.get(serving_endpoint_universal)
    state = endpoint_state.state
    print(f"  State: ready={state.ready} | config_update={state.config_update}")
    if state.ready == EndpointStateReady.READY:
        print(f"\nEndpoint '{serving_endpoint_universal}' is ready!")
        break
    if state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
        raise RuntimeError(f"Endpoint deployment failed: {state}")
    time.sleep(120)

## Create a classification serving endpoint for specific data

Compared to the universal, this serving endpoint has a context dataset already loaded, which means that its inferences are only for test samples relating to that data distribution. Therefore it is not universal.

Benefits:
- Faster inference due to KV caching
- Can be in SQL with the ai function [ai_query](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query), passing only the test samples to infer on

### Create Pyfunc MLflow model

In [0]:

class TabICLClassifierWithContext(mlflow.pyfunc.PythonModel):
    """
    Dataset-specific PyFunc wrapper for TabICL v2 Classifier with KV-cache.

    Usage:
        model = TabICLClassifierWithContext()
        model.fit(X_train, y_train, feature_cols)  # stores data in the instance
        mlflow.pyfunc.log_model(python_model=model, ...)  # data pickled with model

    On the serving endpoint, load_context unpickles the training data and fits
    the classifier with kv_cache=True. Subsequent predict() calls only process
    new test rows, reusing the cached training context.

    Benefits:
        - ai_query compatible: each input row → one output row
        - Faster inference: KV-cache reuses training context
        - No external artifacts or /tmp files — training data is pickled with the model

    Input: DataFrame with feature columns only (no __role__ or __target__ needed)
    Output: DataFrame with prediction (str) and probabilities (JSON str) — same fixed schema as the universal model.
    """

    def fit(self, X_train, y_train, feature_cols):
        """Store training data as instance attributes — pickled with the model."""
        self._X_train = X_train.astype(np.float32)
        self._y_train = y_train
        self._feature_cols = list(feature_cols)
        return self

    def load_context(self, context):
        import torch
        from tabicl import TabICLClassifier

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clf = TabICLClassifier(device=device, kv_cache=True)
        self.clf.fit(self._X_train, self._y_train)
        print(f"Model fitted on {self._X_train.shape[0]} training samples with KV-cache enabled.")

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        # model_input contains ONLY test features — no role/target columns needed
        X_test = model_input.values.astype(np.float32)

        # Fast inference: reuses cached KV projections from training data
        predictions = self.clf.predict(X_test)
        probabilities = self.clf.predict_proba(X_test)

        # Get class labels in the same order as predict_proba columns
        classes = self.clf.classes_

        # Encode probabilities as JSON: {"class_0": 0.85, "class_1": 0.15, ...}
        prob_dicts = [
            json.dumps({str(cls): round(float(p), 6) for cls, p in zip(classes, row)})
            for row in probabilities
        ]

        return pd.DataFrame({
            "prediction": predictions,
            "probabilities": prob_dicts,
        })



### Register model into UC

First, we select one of the sample datasets. This particular serving endpoint will work for this data.  

In [0]:
# Load the Supplier Delay Risk training dataset from Delta table
df_delay = spark.table("supplier_delay_risk_train").toPandas()
# Prepare features - encode categorical columns
df_encoded = pd.get_dummies(df_delay, columns=['supplier_tier', 'supplier_country'], drop_first=True)
# Separate features and target
feature_cols = [col for col in df_encoded.columns if col != 'is_delayed']
X = df_encoded[feature_cols].values
y = df_delay['is_delayed'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [0]:

# Instantiate and fit to bring train data into context
model = TabICLClassifierWithContext()
model.fit(X_train, y_train, feature_cols)

# Same fixed output schema as universal model (class-count agnostic)
output_schema_with_context = Schema([
    ColSpec(DataType.string, "prediction"),
    ColSpec(DataType.string, "probabilities"),  # JSON object: class → prob
])
signature_with_context = ModelSignature(inputs=None, outputs=output_schema_with_context)

# Register model in UC
registered_with_context_name = f"{CATALOG}.{SCHEMA}.tabicl_with_context_supplier_delay"

with mlflow.start_run(run_name="tabicl_with_context_supplier_delay") as run:
    model_info = mlflow.pyfunc.log_model(
        name="tabicl_with_context_supplier_delay",
        python_model=model,
        signature=signature_with_context,
        input_example=pd.DataFrame(X_train[:3], columns=feature_cols),
        pip_requirements=[
            "tabicl",
            "pandas",
            "numpy",
        ],
        registered_model_name=registered_with_context_name,
    )
    print(f"Model URI: {model_info.model_uri}")
    print(f"Registered as: {registered_with_context_name}")


### Create GPU serving endpoint

In [0]:
serving_endpoint_with_context = "tabicl_with_context_supplier_delay"

versions = client.search_model_versions(f"name='{registered_with_context_name}'")
latest_version = str(max(int(v.version) for v in versions))
print(f"Deploying model version {latest_version} of '{registered_with_context_name}'")

endpoint = w.serving_endpoints.create(
    name=serving_endpoint_with_context,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=registered_with_context_name,
                entity_version=latest_version,
                workload_size="Small",
                workload_type=ServingModelWorkloadType.GPU_SMALL,
                scale_to_zero_enabled=True,
            )
        ]
    ),
)
print(f"Endpoint '{serving_endpoint_with_context}' creation triggered.")

In [0]:
# Poll until the serving endpoint is ready
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate

print(f"Waiting for endpoint '{serving_endpoint_with_context}' to be ready...")
while True:
    endpoint_state = w.serving_endpoints.get(serving_endpoint_with_context)
    state = endpoint_state.state
    print(f"  State: ready={state.ready} | config_update={state.config_update}")
    if state.ready == EndpointStateReady.READY:
        print(f"\nEndpoint '{serving_endpoint_with_context}' is ready!")
        break
    if state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
        raise RuntimeError(f"Endpoint deployment failed: {state}")
    time.sleep(120)

## Inference using universal serving endpoint

Like in notebook 01_classification, we make MLflow runs for TabICL, but we use the serving endpoint to generate the inferences and log them into the same MLflow experiment.  

In [0]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

In [0]:
def evaluate_classification(table_name, target, problem_type, task_name,
                            test_size=0.2, stratify=True):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(
        df, target=target, test_size=test_size, stratify=stratify
    )
    n_train, n_test, n_features = len(X_train), len(X_test), X_train.shape[1]

    # Encode categorical columns (TabICL requires numeric inputs)
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train = X_train.copy()
        X_test = X_test.copy()
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
        X_test[cat_cols] = enc.transform(X_test[cat_cols])

    # --- TabICL via universal serving endpoint ---
    with mlflow.start_run(run_name=f"{task_name}_tabicl_serving_endpoints"):
        mlflow.log_params({
            "vendor": "tabicl", "model_type": "TabICLClassifierUniversal",
            "task": task_name, "problem_type": problem_type,
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
            "serving_endpoint": serving_endpoint_universal,
        })

        # Build payload: train rows + test rows
        feature_cols_list = [f"f{i}" for i in range(n_features)] if not hasattr(X_train, 'columns') else list(X_train.columns)
        train_pdf = pd.DataFrame(X_train, columns=feature_cols_list).astype(np.float64).round(2)
        train_pdf["__role__"] = "train"
        train_pdf["__target__"] = y_train.astype(str)

        test_pdf = pd.DataFrame(X_test, columns=feature_cols_list).astype(np.float64).round(2)
        test_pdf["__role__"] = "test"
        test_pdf["__target__"] = ""

        payload = pd.concat([train_pdf, test_pdf], ignore_index=True)

        resp = w.serving_endpoints.query(
            name=serving_endpoint_universal,
            dataframe_records=payload.to_dict(orient="records"),
        )

        # Parse response — cast predictions back to original y_test dtype
        predictions_df = pd.DataFrame(resp.predictions)
        y_pred = predictions_df["prediction"].astype(type(y_test[0])).values
        y_pred_proba = np.array([
            [v for v in json.loads(p).values()]
            for p in predictions_df["probabilities"]
        ])

        metrics = classification_metrics(y_test, y_pred, y_pred_proba)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="tabicl", task=task_name, problem_type=problem_type,
                   model_name="TabICLClassifierUniversal (serving)", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] TabICL (serving): acc={metrics['accuracy']:.4f} "
          f"f1={metrics['f1']:.4f} roc_auc={metrics['roc_auc']}")

    # --- Shared baselines (identical split) ---
    for name, m in train_baselines_classification(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type=problem_type,
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: acc={m['accuracy']:.4f} f1={m['f1']:.4f}")
    return metrics

In [0]:
_ = evaluate_classification(
    table_name="supplier_delay_risk_train",
    target="is_delayed",
    problem_type="binary_classification",
    task_name="supplier_delay_risk",
    test_size=0.2, stratify=True,
)

## Inference example with ai_query

The cells below showcase how ai_query can be used together with the serving endpoint with context to predict new samples for a specific ML task subject to a data distribution. In our case, we created the serving endpoint with the supplier delay risk as context.

In [0]:
# Store test samples into a new delta table to showcase ai_query functionality
df_delay = spark.table("supplier_delay_risk_train").toPandas()
df_encoded = pd.get_dummies(df_delay, columns=['supplier_tier', 'supplier_country'], drop_first=True)
feature_cols = [col for col in df_encoded.columns if col != 'is_delayed']
X = df_encoded[feature_cols].values
y = df_delay['is_delayed'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Store test features as a table for ai_query usage
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_table_name = f"{CATALOG}.{SCHEMA}.supplier_delay_risk_test_ai_query"
spark.createDataFrame(test_df).write.mode("overwrite").saveAsTable(test_table_name)
print(f"Test data saved to {test_table_name} ({len(test_df)} rows)")


In [0]:
%sql
SELECT
  t.*,
  result.prediction,
  result.probabilities
FROM (
  SELECT *,
    ai_query(
      'tabicl_with_context_supplier_delay',
      struct(*),
      returnType => 'STRUCT<prediction:STRING, probabilities:STRING>'
    ) AS result
  FROM (
    SELECT * FROM supplier_delay_risk_test_ai_query LIMIT 10
  )
) t